In [24]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date, time
from math import exp, sqrt

BASE_URL = "http://127.0.0.1:25510/v2"

def td_get(endpoint, params, timeout=180):
    """
    Simple helper to call ThetaData REST API.
    Theta Terminal must be running.

    timeout=180 gives ThetaData up to 3 minutes for large option chains.
    """
    url = BASE_URL + endpoint
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()

    if data.get("header", {}).get("error_type") is not None:
        raise RuntimeError(data["header"])

    return data

In [2]:
def list_expirations(root):
    data = td_get("/list/expirations", {"root": root})
    exps = data["response"]
    return sorted([int(x) for x in exps])

spx_exps = list_expirations("SPX")
spxw_exps = list_expirations("SPXW")

print("SPX expirations:", spx_exps[:5], "...", spx_exps[-5:])
print("SPXW expirations:", spxw_exps[:5], "...", spxw_exps[-5:])


SPX expirations: [20120616, 20120721, 20120818, 20120922, 20121020] ... [20280616, 20281215, 20291221, 20301220, 20311219]
SPXW expirations: [20120601, 20120608, 20120622, 20120706, 20120713] ... [20261120, 20261130, 20261231, 20270331, 20270630]


In [3]:
def yyyymmdd_to_date(x):
    return datetime.strptime(str(x), "%Y%m%d").date()

def choose_vix_expirations(trade_date_yyyymmdd):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)

    candidates = []

    # SPX standard AM-settled options
    for expi in spx_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt:
            candidates.append(("SPX", expi, (exp_dt - trade_dt).days))

    # SPXW weeklies / PM-settled. Keep Fridays for initial VIX-style replication.
    for expi in spxw_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt and exp_dt.weekday() == 4:
            candidates.append(("SPXW", expi, (exp_dt - trade_dt).days))

    candidates = pd.DataFrame(candidates, columns=["root", "exp", "days"])
    candidates = candidates.sort_values("days")

    near = candidates[candidates["days"] <= 30].tail(1)
    next_ = candidates[candidates["days"] > 30].head(1)

    if near.empty or next_.empty:
        raise ValueError("Could not find near and next expirations around 30 days.")

    return pd.concat([near, next_]).reset_index(drop=True)

TRADE_DATE = 20240116
terms = choose_vix_expirations(TRADE_DATE)
terms

,root,exp,days
0,SPXW,20240209,24
1,SPX,20240216,31


In [4]:
QUOTE_COLUMNS = [
    "ms_of_day", "bid_size", "bid_exchange", "bid", "bid_condition",
    "ask_size", "ask_exchange", "ask", "ask_condition", "date"
]

def get_chain_at_time(root, expi, trade_date, calc_time_ms):
    data = td_get(
        "/bulk_at_time/option/quote",
        {
            "root": root,
            "exp": expi,
            "start_date": trade_date,
            "end_date": trade_date,
            "ivl": calc_time_ms
        }
    )

    rows = []

    for item in data["response"]:
        contract = item["contract"]
        ticks = item["ticks"]

        if len(ticks) == 0:
            continue

        # Bulk-at-time should usually return one quote per contract.
        tick = ticks[-1]
        row = dict(zip(QUOTE_COLUMNS, tick))

        row["root"] = contract["root"]
        row["expiration"] = contract["expiration"]
        row["strike"] = contract["strike"] / 1000  # ThetaData strikes are usually strike * 1000
        row["right"] = contract["right"]           # C or P

        rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No data returned for {root} {expi} on {trade_date}")

    df["mid"] = (df["bid"] + df["ask"]) / 2
    return df

CALC_TIME_MS = 57600000  # 16:00 ET

chains = []

for _, row in terms.iterrows():
    chain = get_chain_at_time(
        root=row["root"],
        expi=int(row["exp"]),
        trade_date=TRADE_DATE,
        calc_time_ms=CALC_TIME_MS
    )
    chain["days"] = row["days"]
    chains.append(chain)

raw_options = pd.concat(chains, ignore_index=True)

raw_options.head()

,ms_of_day,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition,date,root,expiration,strike,right,mid,days
0,57600000,15,5,3563.2,50,15,5,3572.00,50,20240116,SPXW,20240209,1200.0,C,3567.600,24
1,57600000,0,5,0.0,50,570,5,0.05,50,20240116,SPXW,20240209,1200.0,P,0.025,24
2,57600000,15,5,3364.0,50,15,5,3372.80,50,20240116,SPXW,20240209,1400.0,C,3368.400,24
3,57600000,0,5,0.0,50,565,5,0.05,50,20240116,SPXW,20240209,1400.0,P,0.025,24
4,57600000,15,5,3164.7,50,15,5,3173.50,50,20240116,SPXW,20240209,1600.0,C,3169.100,24


In [5]:
def calc_single_term_variance(chain, minutes_to_expiry, r=0.05):
    """
    Implements the core VIX single-expiration variance calculation.

    chain: option chain for one expiration, with columns:
           strike, right, bid, ask, mid
    minutes_to_expiry: minutes from calculation time to option expiry
    r: continuously compounded risk-free rate
    """

    T = minutes_to_expiry / 525600  # calendar years, 365 * 1440

    # Keep clean quotes
    df = chain.copy()
    df = df[(df["bid"] >= 0) & (df["ask"] > 0) & (df["ask"] >= df["bid"])]
    df = df[["strike", "right", "bid", "ask", "mid"]]

    calls = df[df["right"] == "C"].set_index("strike")
    puts  = df[df["right"] == "P"].set_index("strike")

    common_strikes = calls.index.intersection(puts.index)
    cp = pd.DataFrame({
        "call_mid": calls.loc[common_strikes, "mid"],
        "put_mid": puts.loc[common_strikes, "mid"],
        "call_bid": calls.loc[common_strikes, "bid"],
        "put_bid": puts.loc[common_strikes, "bid"],
        "call_ask": calls.loc[common_strikes, "ask"],
        "put_ask": puts.loc[common_strikes, "ask"],
    }).sort_index()

    # ATM strike = strike where |call - put| is smallest
    cp["abs_diff"] = (cp["call_mid"] - cp["put_mid"]).abs()
    atm_strike = cp["abs_diff"].idxmin()

    # Forward from put-call parity
    F = atm_strike + exp(r * T) * (cp.loc[atm_strike, "call_mid"] - cp.loc[atm_strike, "put_mid"])

    # K0 = strike immediately below forward
    strikes = np.array(sorted(common_strikes))
    k0_candidates = strikes[strikes <= F]
    if len(k0_candidates) == 0:
        raise ValueError("No K0 found below forward.")

    K0 = k0_candidates[-1]

    # Build OTM option set
    selected_rows = []

    # Puts below K0
    put_strikes = sorted([k for k in puts.index if k < K0], reverse=True)
    zero_count = 0
    for k in put_strikes:
        bid = puts.loc[k, "bid"]
        ask = puts.loc[k, "ask"]
        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue
        zero_count = 0
        selected_rows.append({"strike": k, "Q": puts.loc[k, "mid"]})

    # Calls above K0
    call_strikes = sorted([k for k in calls.index if k > K0])
    zero_count = 0
    for k in call_strikes:
        bid = calls.loc[k, "bid"]
        ask = calls.loc[k, "ask"]
        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue
        zero_count = 0
        selected_rows.append({"strike": k, "Q": calls.loc[k, "mid"]})

    # K0 uses average of call and put midpoint
    k0_q = 0.5 * (calls.loc[K0, "mid"] + puts.loc[K0, "mid"])
    selected_rows.append({"strike": K0, "Q": k0_q})

    selected = pd.DataFrame(selected_rows).drop_duplicates("strike").sort_values("strike").reset_index(drop=True)

    # Delta K
    selected["delta_k"] = np.nan
    for i in range(len(selected)):
        if i == 0:
            selected.loc[i, "delta_k"] = selected.loc[i + 1, "strike"] - selected.loc[i, "strike"]
        elif i == len(selected) - 1:
            selected.loc[i, "delta_k"] = selected.loc[i, "strike"] - selected.loc[i - 1, "strike"]
        else:
            selected.loc[i, "delta_k"] = (selected.loc[i + 1, "strike"] - selected.loc[i - 1, "strike"]) / 2

    selected["contribution"] = (
        selected["delta_k"] / (selected["strike"] ** 2)
        * exp(r * T)
        * selected["Q"]
    )

    variance = (2 / T) * selected["contribution"].sum() - (1 / T) * ((F / K0) - 1) ** 2

    return {
        "variance": variance,
        "vol": sqrt(max(variance, 0)) * 100,
        "T": T,
        "minutes": minutes_to_expiry,
        "F": F,
        "K0": K0,
        "selected_options": selected
    }

In [6]:
def minutes_to_expiry_approx(trade_date_yyyymmdd, exp_yyyymmdd, calc_time_ms):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes = calc_time_ms / 1000 / 60
    expiry_minutes = 16 * 60  # 4:00 PM ET approximation

    calendar_days = (exp_dt - trade_dt).days
    return int(calendar_days * 1440 + expiry_minutes - calc_minutes)

term_results = []

for _, row in terms.iterrows():
    root = row["root"]
    expi = int(row["exp"])

    chain = raw_options[
        (raw_options["root"] == root) &
        (raw_options["expiration"] == expi)
    ]

    mins = minutes_to_expiry_approx(TRADE_DATE, expi, CALC_TIME_MS)

    result = calc_single_term_variance(chain, mins, r=0.05)

    term_results.append({
        "root": root,
        "exp": expi,
        "days": row["days"],
        "minutes": mins,
        "variance": result["variance"],
        "vol": result["vol"],
        "F": result["F"],
        "K0": result["K0"]
    })

term_df = pd.DataFrame(term_results)
term_df

,root,exp,days,minutes,variance,vol,F,K0
0,SPXW,20240209,24,34560,0.018789,13.707176,4781.504940,4780.0
1,SPX,20240216,31,44640,0.018998,13.783338,4783.694468,4780.0


In [7]:
def interpolate_to_30d(term_df):
    near = term_df.iloc[0]
    next_ = term_df.iloc[1]

    M1 = near["minutes"]
    M2 = next_["minutes"]
    M30 = 30 * 1440
    M365 = 365 * 1440

    T1 = M1 / M365
    T2 = M2 / M365

    var1 = near["variance"]
    var2 = next_["variance"]

    # Cboe-style interpolation in variance-time
    variance_30d = (
        T1 * var1 * ((M2 - M30) / (M2 - M1)) +
        T2 * var2 * ((M30 - M1) / (M2 - M1))
    ) * (M365 / M30)

    vix = 100 * sqrt(max(variance_30d, 0))
    return vix

my_vix = interpolate_to_30d(term_df)
my_vix

13.774654750579229

In [10]:
import pandas as pd

cboe_url = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"

cboe_vix = pd.read_csv(cboe_url)

print(cboe_vix.head())
print(cboe_vix.columns)

         DATE   OPEN   HIGH    LOW  CLOSE
0  01/02/1990  17.24  17.24  17.24  17.24
1  01/03/1990  18.19  18.19  18.19  18.19
2  01/04/1990  19.22  19.22  19.22  19.22
3  01/05/1990  20.11  20.11  20.11  20.11
4  01/08/1990  20.26  20.26  20.26  20.26
Index(['DATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE'], dtype='object')


In [13]:
# Convert dates and find our trade date

cboe_vix["DATE"] = pd.to_datetime(cboe_vix["DATE"])

target_date = pd.to_datetime(str(TRADE_DATE), format="%Y%m%d")

official_row = cboe_vix[cboe_vix["DATE"] == target_date]

display(official_row)

,DATE,OPEN,HIGH,LOW,CLOSE
8584,2024-01-16,14.12,14.35,13.52,13.84


In [14]:
official_vix_close = float(official_row["CLOSE"].iloc[0])

print("My replicated VIX:", round(my_vix, 4))
print("Official Cboe VIX close:", round(official_vix_close, 4))
print("Difference:", round(my_vix - official_vix_close, 4))
print("Absolute difference:", round(abs(my_vix - official_vix_close), 4))

My replicated VIX: 13.7747
Official Cboe VIX close: 13.84
Difference: -0.0653
Absolute difference: 0.0653


In [15]:
def calculate_vix_for_date(trade_date, calc_time_ms=57600000, r=0.05):
    """
    Calculate a VIX-like 30-day implied volatility measure for one trade date.

    trade_date: YYYYMMDD integer, example 20240116
    calc_time_ms: milliseconds after midnight ET. 57600000 = 16:00:00 ET
    r: flat continuously compounded risk-free rate, temporary simplification
    """

    # 1. Choose near and next expirations around 30 days
    terms = choose_vix_expirations(trade_date)

    # 2. Pull option chains
    chains = []

    for _, row in terms.iterrows():
        print("Pulling:", row["root"], int(row["exp"]))

        chain = get_chain_at_time(
            root=row["root"],
            expi=int(row["exp"]),
            trade_date=trade_date,
            calc_time_ms=calc_time_ms
        )

        chain["days"] = row["days"]
        chains.append(chain)

    raw_options = pd.concat(chains, ignore_index=True)

    # 3. Calculate variance for each expiration
    term_results = []

    for _, row in terms.iterrows():
        root = row["root"]
        expi = int(row["exp"])

        chain = raw_options[
            (raw_options["root"] == root) &
            (raw_options["expiration"] == expi)
        ]

        mins = minutes_to_expiry_approx(trade_date, expi, calc_time_ms)

        result = calc_single_term_variance(chain, mins, r=r)

        term_results.append({
            "root": root,
            "exp": expi,
            "days": row["days"],
            "minutes": mins,
            "variance": result["variance"],
            "vol": result["vol"],
            "F": result["F"],
            "K0": result["K0"],
            "num_selected_options": len(result["selected_options"])
        })

    term_df = pd.DataFrame(term_results)

    # 4. Interpolate to 30 days
    vix_value = interpolate_to_30d(term_df)

    return {
        "trade_date": trade_date,
        "calc_time_ms": calc_time_ms,
        "vix": vix_value,
        "term_df": term_df,
        "raw_options": raw_options
    }

In [16]:
result = calculate_vix_for_date(20240116)

print("Calculated VIX:", round(result["vix"], 4))

display(result["term_df"])

Pulling: SPXW 20240209
Pulling: SPX 20240216
Calculated VIX: 13.7747


,root,exp,days,minutes,variance,vol,F,K0,num_selected_options
0,SPXW,20240209,24,34560,0.018789,13.707176,4781.504940,4780.0,161
1,SPX,20240216,31,44640,0.018998,13.783338,4783.694468,4780.0,351


In [17]:
test_dates = [
    20240116,
    20240215,
    20240315,
    20240415,
    20240515
]

vix_results = []

for d in test_dates:
    print("\n====================")
    print("Calculating:", d)

    try:
        result = calculate_vix_for_date(d)

        vix_results.append({
            "date": d,
            "replicated_vix": result["vix"],
            "near_root": result["term_df"].iloc[0]["root"],
            "near_exp": result["term_df"].iloc[0]["exp"],
            "near_days": result["term_df"].iloc[0]["days"],
            "near_vol": result["term_df"].iloc[0]["vol"],
            "next_root": result["term_df"].iloc[1]["root"],
            "next_exp": result["term_df"].iloc[1]["exp"],
            "next_days": result["term_df"].iloc[1]["days"],
            "next_vol": result["term_df"].iloc[1]["vol"]
        })

    except Exception as e:
        print("FAILED:", d, e)
        vix_results.append({
            "date": d,
            "replicated_vix": np.nan,
            "error": str(e)
        })

vix_results_df = pd.DataFrame(vix_results)

display(vix_results_df)


Calculating: 20240116
Pulling: SPXW 20240209
Pulling: SPX 20240216

Calculating: 20240215
Pulling: SPXW 20240315
Pulling: SPXW 20240322

Calculating: 20240315
Pulling: SPXW 20240412
Pulling: SPX 20240419
FAILED: 20240315 HTTPConnectionPool(host='127.0.0.1', port=25510): Read timed out. (read timeout=60)

Calculating: 20240415
Pulling: SPXW 20240510
Pulling: SPX 20240517

Calculating: 20240515
Pulling: SPXW 20240614
Pulling: SPXW 20240621


,date,replicated_vix,near_root,near_exp,near_days,near_vol,next_root,next_exp,next_days,next_vol,error
0,20240116,13.774655,SPXW,20240209.0,24.0,13.707176,SPX,20240216.0,31.0,13.783338,NaN
1,20240215,14.259789,SPXW,20240315.0,29.0,14.187315,SPXW,20240322.0,36.0,14.605011,NaN
2,20240315,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"HTTPConnectionPool(host='127.0.0.1', port=2551..."
3,20240415,19.109972,SPXW,20240510.0,25.0,19.374667,SPX,20240517.0,32.0,19.026499,NaN
4,20240515,12.389784,SPXW,20240614.0,30.0,12.389784,SPXW,20240621.0,37.0,12.350464,NaN


In [19]:
# Make sure Cboe VIX history is loaded
cboe_url = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"
cboe_vix = pd.read_csv(cboe_url)
cboe_vix["DATE"] = pd.to_datetime(cboe_vix["DATE"])

# Prepare our replicated results
compare_df = vix_results_df.copy()
compare_df["DATE"] = pd.to_datetime(compare_df["date"].astype(str), format="%Y%m%d")

# Merge with official Cboe daily closes
compare_df = compare_df.merge(
    cboe_vix[["DATE", "CLOSE"]],
    on="DATE",
    how="left"
)

compare_df = compare_df.rename(columns={"CLOSE": "official_vix_close"})

compare_df["difference"] = compare_df["replicated_vix"] - compare_df["official_vix_close"]
compare_df["abs_difference"] = compare_df["difference"].abs()

display(compare_df[[
    "date",
    "replicated_vix",
    "official_vix_close",
    "difference",
    "abs_difference",
    "near_exp",
    "near_days",
    "next_exp",
    "next_days"
]])

,date,replicated_vix,official_vix_close,difference,abs_difference,near_exp,near_days,next_exp,next_days
0,20240116,13.774655,13.84,-0.065345,0.065345,20240209.0,24.0,20240216.0,31.0
1,20240215,14.259789,14.01,0.249789,0.249789,20240315.0,29.0,20240322.0,36.0
2,20240315,NaN,14.41,NaN,NaN,NaN,NaN,NaN,NaN
3,20240415,19.109972,19.23,-0.120028,0.120028,20240510.0,25.0,20240517.0,32.0
4,20240515,12.389784,12.45,-0.060216,0.060216,20240614.0,30.0,20240621.0,37.0


In [25]:
result_debug = calculate_vix_for_date(20240315)

print("Calculated VIX:", round(result_debug["vix"], 4))
display(result_debug["term_df"])

Pulling: SPXW 20240412
Pulling: SPX 20240419
Calculated VIX: 14.5136


,root,exp,days,minutes,variance,vol,F,K0,num_selected_options
0,SPXW,20240412,28,40320,0.021014,14.496202,5138.343659,5135.0,181
1,SPX,20240419,35,50400,0.021166,14.548483,5143.241589,5140.0,370
